In [1]:
import tqdm

In [2]:
import re

In [3]:
import json

In [4]:
import pickle

In [5]:
from dotenv import load_dotenv

In [6]:
load_dotenv()

True

In [7]:
import pandas as pd

In [8]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [9]:
MODEL_ID = "google/gemma-4-E2B-it"

In [10]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [11]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [12]:
with open("titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags = pickle.load(f)

In [13]:
len(titles_with_tags)

1187

In [14]:
data = []

In [15]:
for title, tags in titles_with_tags.items():
    for tag in tags:
        data.append((title, tag))

In [16]:
data[0]

('Исследование приоритетов и механизмов реализации отраслевых (секторальных) политик в избранных странах БРИКС (членах и кандидатах), во взаимосвязи с повесткой международных доноров',
 'international_relations')

In [17]:
data[-1]

('Обзор Методологии в Области Регионоведения. Этап сбора данных 1.',
 'regional_studies')

In [18]:
df = pd.DataFrame(data=data, columns=["title_rus", "tag"])

In [19]:
df

,title_rus,tag
0,Исследование приоритетов и механизмов реализац...,international_relations
1,Исследование приоритетов и механизмов реализац...,political_economics
2,Исследование приоритетов и механизмов реализац...,policy_analysis
3,Исследование приоритетов и механизмов реализац...,geopolitics
4,Исследование приоритетов и механизмов реализац...,BRICS
...,...,...
6035,Актив Центра развития карьеры - профориентацио...,student_services
6036,Обзор Методологии в Области Регионоведения. Эт...,geography
6037,Обзор Методологии в Области Регионоведения. Эт...,methodology
6038,Обзор Методологии в Области Регионоведения. Эт...,data_collection


In [20]:
dfgr = df.groupby(by=["tag"]).agg({"title_rus": "nunique"}).reset_index()
dfgr.columns = ["tag", "count"]
dfgr = dfgr.sort_values(by=["count"], ascending=False).reset_index(drop=True)
dfgr.head(32)

,tag,count
0,international_relations,97
1,sociology,96
2,economics,93
3,linguistics,92
4,software_engineering,89
5,marketing,85
6,cultural_studies,76
7,education,74
8,history,73
9,media_studies,70


In [21]:
selected_tags = dfgr.head(50)["tag"].to_list()

In [22]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can create artificial user profiles based on student project titles in russian.
Please, analyze given list of student project titles in russian and create several possible user profiles (up to 4) with own preferences in provided field of science.
Return stricly JSON output. Be brief.
For example.
Input data:
```json
{"machine_learning": ["Исследование методов решения задачи линейной регрессии", "Исследование методов решения задачи линейной классификации", "Исследование методов решения задачи обучения с подкреплением"]}
```
Your output:
```json
{"machine_learning_researcher_classic_algorithms": "A user, which is interested in machine learning and classic algorithms in particular", "machine_learning_researcher_reinforcement_learning": "A user, which is interested in machine learning and reinforcement learning in particular"}
```
"""

In [23]:
messages = []

In [24]:
for tag in selected_tags:
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {tag: list(df[df["tag"] == tag]["title_rus"].unique())},
                    ensure_ascii=False,
                ),
            },
        )
    )

In [25]:
messages[0][1]["content"]

'{"international_relations": ["Исследование приоритетов и механизмов реализации отраслевых (секторальных) политик в избранных странах БРИКС (членах и кандидатах), во взаимосвязи с повесткой международных доноров", "Сеть военно-политических союзов в Евразии: база данных", "Запуск регулярного подкаста НИУ ВШЭ о странах Азии", "Студенческий клуб \\"Говорим об Эфиопии\\"", "Визовое сопровождение иностранных студентов", "Международный мониторинг ответственного бизнеса", "Школа будущего международника: десятый сезон (2025/2026)", "Внешняя политика России в отношении стран Латинской Америки в XXI веке", "Ближневосточный клуб НИУ ВШЭ: платформа для изучения региона и развития академического и карьерного потенциала студентов-ближневосточников 2025/26", "Японский клуб «Мусуби» НИУ ВШЭ", "Практикум Совета молодых дипломатов", "Новости Ибероамерики-II", "Итальянский клуб ВШЭ", "Исследование правового обеспечения информационной безопасности стран БРИКС", "VII Международная конференция «Мировое боль

In [26]:
len(messages[0][1]["content"])

7586

In [27]:
answers = {}

In [28]:
for tag, message in tqdm.tqdm(list(zip(selected_tags, messages))):
    try:
        text = processor.apply_chat_template(
            message, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )

        inputs = processor(text=text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(**inputs, max_new_tokens=2048)
        response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

        json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
        if json_match:
            json_str = json_match.group(1)
        else:
            json_str = response

        print(json_str)
        answers.update(json.loads(json_str))
    except Exception as e:
        print(f"something went wrong with tag {tag} - {e}")

  2%|▏         | 1/50 [00:28<23:06, 28.29s/it]

{
  "international_relations_geopolitics_expert": "A user deeply interested in complex geopolitical dynamics, focusing on BRICS, Eurasia, China-Russia relations, energy politics, and regional conflicts (Middle East, Central Asia). Prefers research into strategic interests and policy mechanisms.",
  "international_relations_regional_studies_enthusiast": "A user focused on regional studies, particularly East Asia (Japan, Korea, China, Vietnam) and South Asia. Interested in cultural exchange, specific bilateral relationships, and regional blocs (like the 'Trio' of Northeast Asia).",
  "international_relations_law_and_security_analyst": "A user interested in the legal and security aspects of international relations, including information security, dispute resolution mechanisms (EAEU), and the role of international law in digital diplomacy.",
  "international_relations_student_and_ambassador": "A student or young professional actively involved in student clubs, networking, media management 

  4%|▍         | 2/50 [00:54<21:26, 26.80s/it]

{"sociology_culture_and_identity_expert": "A user deeply interested in cultural studies, identity formation, social memory, and the intersection of culture, politics, and media. Prefers qualitative research, ethnography, and critical analysis of cultural phenomena.", "intercultural_communication_and_global_studies": "A user focused on cross-cultural communication, international relations, and comparative studies (e.g., Russia/Mexico, various Asian codes). Interested in global trends, political sociology, and comparative cultural practices.", "social_and_political_theory_researcher": "A user leaning towards theoretical sociology, focusing on large-scale societal structures, political elites, social inequality, and the impact of technology (AI) on society. Seeks high-level theoretical understanding.", "social_and_youth_issues_specialist": "A user interested in contemporary social issues, including youth agency, mental health (e.g., HIV, bullying), social entrepreneurship, and digital cul

  6%|▌         | 3/50 [01:19<20:25, 26.07s/it]

...

{"social_media_strategist_education": "A user highly focused on social media marketing within the educational sector, specializing in promoting master's programs, university labs, and specialized courses.", "content_creator_and_niche_marketing": "A user interested in developing diverse content strategies across various social media platforms (Telegram, VK) for niche communities, academic programs, and specific topics (e.g., linguistics, media management).", "brand_and_community_manager": "A user skilled in managing social media presence for brands, clubs, and community groups, focusing on engagement and building community around specific projects.", "SMM_for_career_and_recruitment": "A user interested in using social media for professional purposes, such as recruiting students, finding job opportunities, and marketing HR-related programs."}


 98%|█████████▊| 49/50 [15:28<00:16, 16.29s/it]

{
  "social_worker_community_organizer": "A user focused on community building, youth engagement, and developing social and cultural groups (e.g., 'STEPS COMMUNITY', group self-help). Interested in organizing events and community initiatives.",
  "social_worker_specialist_in_vulnerable_populations": "A user deeply interested in supporting specific vulnerable groups, particularly those affected by HIV, children from specific regions (Asia/Middle East), and individuals with dependencies. Focused on direct aid and advocacy.",
  "social_worker_policy_and_legal_advocate": "A user with an interest in the intersection of social work and policy/law, including legal aid, combating stigma (media representation), and developing youth policy/education programs.",
  "social_worker_researcher_and_methodologist": "A user focused on the theoretical and methodological aspects of social work, including developing research methodologies for family systems and analyzing professional dilemmas within the so

100%|██████████| 50/50 [15:39<00:00, 18.79s/it]

{"higher_education_international_relations": "A user focused on international higher education, student mobility, and international conferences.", "higher_education_program_management": "A user interested in developing and promoting university programs (Master's, Bachelor's) and academic strategy.", "academic_support_and_mentoring": "A user interested in student support, academic advising, mentorship, and creating effective learning environments.", "university_promotion_and_outreach": "A user focused on public relations, social media promotion of research labs, and institutional outreach."}


In [29]:
len(answers)

200

In [30]:
answers

{'international_relations_geopolitics_expert': 'A user deeply interested in complex geopolitical dynamics, focusing on BRICS, Eurasia, China-Russia relations, energy politics, and regional conflicts (Middle East, Central Asia). Prefers research into strategic interests and policy mechanisms.',
 'international_relations_regional_studies_enthusiast': "A user focused on regional studies, particularly East Asia (Japan, Korea, China, Vietnam) and South Asia. Interested in cultural exchange, specific bilateral relationships, and regional blocs (like the 'Trio' of Northeast Asia).",
 'international_relations_law_and_security_analyst': 'A user interested in the legal and security aspects of international relations, including information security, dispute resolution mechanisms (EAEU), and the role of international law in digital diplomacy.',
 'international_relations_student_and_ambassador': 'A student or young professional actively involved in student clubs, networking, media management (socia

In [31]:
with open("artificial_profiles.json", "w") as f:
    json.dump(answers, f)